### The objective of this notebook is to implement LightGBM global forecasting model, and compare the results with those of statistical models. Unlike the statistical models, LightGBM will be trained across the entire 228 series simultaneously, like a tabular dataset along with some engineered features. 

In [2]:
# Importing the libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder

from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import MinTrace

In [3]:
# Loading the hierarchy ready dataset

df = pd.read_csv('../data/processed/hierarchy_ready.csv')
df['ds'] = pd.to_datetime(df['ds'])

# Dropping the 4 series
to_drop = {
    'TOTAL/MUMBAI/MUMBAI-RAJKOT',
    'TOTAL/DELHI/DELHI-RAJKOT',
    'TOTAL/DEHRA DUN/DEHRA DUN-DELHI',
    'TOTAL/DEHRA DUN'
}

df = df[~df['unique_id'].isin(to_drop)].copy()

# Sorting the values for lag construction
df = df.sort_values(['unique_id','ds']).reset_index(drop = True)

print('Shape: ',df.shape)
print('Series: ',df['unique_id'].nunique())
print('Date Range: ',df['ds'].min().date(),' to ',df['ds'].max().date())
print(df.head())

Shape:  (26677, 3)
Series:  228
Date Range:  2015-04-01  to  2026-02-01
  unique_id         ds          y
0     TOTAL 2015-04-01  2364483.0
1     TOTAL 2015-05-01  2547864.0
2     TOTAL 2015-06-01  2327666.0
3     TOTAL 2015-07-01  2431149.0
4     TOTAL 2015-08-01  2381990.0


In [4]:
# Feature Engineering 01 - Lag
for lag in [1,2,3,12]:
    df[f'y_lag_{lag}'] = df.groupby('unique_id')['y'].shift(lag)

# check for NaN values in lag columns
print("NaN count after lag feature construction: ")
print(df[['y_lag_1','y_lag_2','y_lag_3','y_lag_12']].isna().sum())

NaN count after lag feature construction: 
y_lag_1      228
y_lag_2      456
y_lag_3      684
y_lag_12    2736
dtype: int64


In [8]:
# Rolling means per series
df['y_rollmean_3'] = (df.groupby('unique_id')['y'].transform(lambda s: s.shift(1).rolling(window = 3, min_periods = 1).mean()))
df['y_rollmean_12'] = (df.groupby('unique_id')['y'].transform(lambda s: s.shift(1).rolling(window = 12, min_periods = 1).mean()))

# Sanity checks
print("NaN counts after rolling means: ")
print(df[['y_rollmean_3','y_rollmean_12']].isna().sum())

# Re-verify on MUMBAI
sample = df[df['unique_id'] == 'TOTAL/MUMBAI'].head(5)[['ds','y','y_rollmean_3']]
print('MUMBAI airport first 5 rows: ')
print(sample.to_string(index = False))

NaN counts after rolling means: 
y_rollmean_3     228
y_rollmean_12    228
dtype: int64
MUMBAI airport first 5 rows: 
        ds        y  y_rollmean_3
2015-04-01 672098.0           NaN
2015-05-01 712385.0 672098.000000
2015-06-01 599957.0 692241.500000
2015-07-01 656874.0 661480.000000
2015-08-01 640882.0 656405.333333


In [9]:
# Calendar Features
df['month'] = df['ds'].dt.month
df['quarter'] = df['ds'].dt.quarter
df['year'] = df['ds'].dt.year

print(df[['ds','month','quarter','year']].head())

          ds  month  quarter  year
0 2015-04-01      4        2  2015
1 2015-05-01      5        2  2015
2 2015-06-01      6        2  2015
3 2015-07-01      7        3  2015
4 2015-08-01      8        3  2015


In [11]:
# Hierarchy level encoding
df['level'] = df['unique_id'].apply(
    lambda x: 0 if '/' not in x
    else (1 if x.count('/') == 1 else 2)
)

# Label encoding for unique id
le = LabelEncoder()
df['series_id'] = le.fit_transform(df['unique_id'])

print("Rows per level: ")
print(df['level'].value_counts().sort_index())
print(f"Unique series_id values: {df['series_id'].nunique()}")
print(df[['unique_id','level','series_id']].head())

Rows per level: 
level
0      131
1     3591
2    22955
Name: count, dtype: int64
Unique series_id values: 228
  unique_id  level  series_id
0     TOTAL      0          0
1     TOTAL      0          0
2     TOTAL      0          0
3     TOTAL      0          0
4     TOTAL      0          0


In [12]:
# Dropping NaN features
df_ml = df.dropna().reset_index(drop = True)
print(f"Rows before dropping NaN values: {len(df)}")
print(f"Rows after dropping NaN values: {len(df_ml)}")

Rows before dropping NaN values: 26677
Rows after dropping NaN values: 23941


In [20]:
# Test and Train Split
split_date = pd.Timestamp('2025-03-01')

train_ml = df_ml[df_ml['ds'] < split_date].copy()
test_ml = df_ml[df_ml['ds'] >= split_date].copy()

print(f"Train: {train_ml.shape} | Range: {train_ml['ds'].min().date()} to {train_ml['ds'].max().date()}")
print(f"Test: {test_ml.shape} | Range: {test_ml['ds'].min().date()} to {test_ml['ds'].max().date()}")

print(f"Series in Train dataset: {train_ml['unique_id'].nunique()}")
print(f"Series in Test dataset: {test_ml['unique_id'].nunique()}")

Train: (21205, 14) | Range: 2016-04-01 to 2025-02-01
Test: (2736, 14) | Range: 2025-03-01 to 2026-02-01
Series in Train dataset: 228
Series in Test dataset: 228


In [27]:
# Final checks for data sanilty
features = ['y_lag_1','y_lag_2','y_lag_3','y_lag_12',
            'y_rollmean_3','y_rollmean_12',
            'month','quarter','year',
            'level','series_id']

# Check for NaN Values
print(train_ml[features].isna().sum().sum(), " NaN values in training dataset")
print(test_ml[features].isna().sum().sum(), " NaN values in test dataset")

# Stats per feature
print(train_ml[features].describe().round(2).to_string())

# Traget column
print(f"Target y: {train_ml['ds'].min().date()} to {train_ml['ds'].max().date()}")
print(f"Test y: {test_ml['ds'].min().date()} to {test_ml['ds'].max().date()}")


0  NaN values in training dataset
0  NaN values in test dataset
          y_lag_1     y_lag_2     y_lag_3    y_lag_12  y_rollmean_3  y_rollmean_12     month   quarter      year     level  series_id
count    21205.00    21205.00    21205.00    21205.00      21205.00       21205.00  21205.00  21205.00  21205.00  21205.00   21205.00
mean     61474.79    61083.44    60701.76    57594.39      61086.67       59526.41      6.54      2.52   2020.58      1.85     114.68
std     318588.08   316367.54   313918.60   295010.77     314767.64      300294.48      3.46      1.12      2.53      0.37      66.37
min          0.00        0.00        0.00        0.00          0.00         141.92      1.00      1.00   2016.00      0.00       0.00
25%       6746.00     6653.50     6577.00     5958.00       6756.33        6870.00      4.00      2.00   2019.00      2.00      57.00
50%      14641.00    14537.00    14446.00    13683.00      14600.33       14424.92      7.00      3.00   2021.00      2.00     115.0

In [28]:
# Saving the files
train_ml.to_csv('../outputs/lightgbm_train.csv',index = False)
test_ml.to_csv('../outputs/lightgbm_test.csv',index = False)